# just_tx vs just_tx_hal：丢包扫速与 ~5% 工作点 RTT

同一块 ESP32-C3、同一 AP、同一台 TR3000。HAL / XDP 用**同一组** TR3000 发包速率扫丢包与回包 PPS，再在各自「丢包≈5%」的最大速率上测 RTT。

| 臂 | 固件 / 冷表 | 回发路径 |
|---|---|---|
| **HAL** | `just_tx_hal` | 应用层 UDP:8080 原载荷回发 |
| **XDP** | `just_drop` 固件 + 冷表 `just_tx.elf` | Wi-Fi 入口 MAC 交换后 `XDP_TX` |

工作点：扫速各点取试次中位丢包；落在 4%–7% 内取最大 `target_pps`，否则取最接近 5% 的点。

数据：`formal.just_tx_rtt_*`，`source_note = xpost-test just-tx-rtt`，`phase` = `sweep` | `operating`。

**复现**（cwd = `xpost/`）：

```bash
cargo xtask agent --listen 0.0.0.0:9240 --no-monitor
```

TR3000：

```bash
XPOST_AGENT_ENDPOINT=192.168.110.96:9240 ./xpost-test just-tx-rtt
```

本笔记只读库（`.env` 设 `PGSCHEMA=formal`）。满速处理 PPS 见 `just_tx_pps.ipynb`。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from utils.db import dsn_summary
from utils.experiment import figure_tx_rtt_loss_vs_rate, load_just_tx_rtt_runs

SOURCE_NOTE = "xpost-test just-tx-rtt"
plt.ioff()
plt.close("all")
print(dsn_summary())

sweep = load_just_tx_rtt_runs(run_kind="pilot", source_note=SOURCE_NOTE, phase="sweep")
assert len(sweep), "无 sweep 行：先跑 ./xpost-test just-tx-rtt（勿加 --no-pg）"

rates = sorted(sweep["target_pps"].unique())


def _med_loss(variant: str, rate: float):
    sub = sweep[(sweep["variant"] == variant) & (np.isclose(sweep["target_pps"], rate))]
    if sub.empty:
        return None
    return float(sub["loss_rate"].median())


hal_loss = [_med_loss("hal", r) for r in rates]
xdp_loss = [_med_loss("xdp", r) for r in rates]
print("rates=", [int(r) for r in rates])
print("hal_loss%=", [None if v is None else round(100 * v, 2) for v in hal_loss])
print("xdp_loss%=", [None if v is None else round(100 * v, 2) for v in xdp_loss])

fig, _ = figure_tx_rtt_loss_vs_rate(
    rates_pps=rates,
    hal_loss=hal_loss,
    xdp_loss=xdp_loss,
    title="Loss vs send rate",
)
display(fig)
plt.close(fig)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from utils.db import dsn_summary
from utils.experiment import (
    figure_tx_rtt_boxplot,
    load_just_tx_rtt_runs,
    load_just_tx_rtt_samples,
)

SOURCE_NOTE = "xpost-test just-tx-rtt"
LOSS_LO, LOSS_HI, TARGET = 0.04, 0.07, 0.05
plt.ioff()
plt.close("all")
print(dsn_summary())

sweep = load_just_tx_rtt_runs(run_kind="pilot", source_note=SOURCE_NOTE, phase="sweep")
ops = load_just_tx_rtt_runs(run_kind="pilot", source_note=SOURCE_NOTE, phase="operating")
assert len(ops), "无 operating 行：先跑 ./xpost-test just-tx-rtt"


def pick_op(variant: str):
    sub = sweep[sweep["variant"] == variant]
    rows = []
    for rate, g in sub.groupby("target_pps"):
        rows.append((float(rate), float(g["loss_rate"].median())))
    in_band = [r for r in rows if LOSS_LO <= r[1] <= LOSS_HI]
    if in_band:
        rate, loss = max(in_band, key=lambda x: x[0])
    else:
        rate, loss = min(rows, key=lambda x: abs(x[1] - TARGET))
    return rate, loss


def rtts_for(variant: str):
    sub = ops[ops["variant"] == variant]
    samples = load_just_tx_rtt_samples(sub["run_id"].tolist())
    ok = samples[samples["ok"] & samples["rtt_ms"].notna()]
    return [float(x) for x in ok["rtt_ms"].tolist()]


hal_rate, hali_loss = pick_op("hal")
xdp_rate, xdp_loss = pick_op("xdp")
hal_rtts = rtts_for("hal")
xdp_rtts = rtts_for("xdp")
hal_label = f"HAL · {hal_rate:.0f} pps · loss {100 * hali_loss:.1f}%"
xdp_label = f"XDP · {xdp_rate:.0f} pps · loss {100 * xdp_loss:.1f}%"
print(hal_label, f"n_rtt={len(hal_rtts)}")
print(xdp_label, f"n_rtt={len(xdp_rtts)}")
assert hali_rtts and xdp_rtts, "operating 样本无成功 RTT"

fig, _ = figure_tx_rtt_boxplot(
    hali_rtts_ms=hal_rtts,
    xdp_rtts_ms=xdp_rtts,
    hali_label=hal_label,
    xdp_label=xdp_label,
)
display(fig)
plt.close(fig)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from utils.db import dsn_summary
from utils.experiment import figure_tx_rtt_return_pps, load_just_tx_rtt_runs

SOURCE_NOTE = "xpost-test just-tx-rtt"
plt.ioff()
plt.close("all")
print(dsn_summary())

sweep = load_just_tx_rtt_runs(run_kind="pilot", source_note=SOURCE_NOTE, phase="sweep")
assert len(sweep), "无 sweep 行：先跑 ./xpost-test just-tx-rtt"

rates = sorted(sweep["target_pps"].unique())


def _med_rpps(variant: str, rate: float):
    sub = sweep[(sweep["variant"] == variant) & (np.isclose(sweep["target_pps"], rate))]
    if sub.empty:
        return None
    return float(sub["return_pps"].median())


hal_rpps = [_med_rpps("hal", r) for r in rates]
xdp_rpps = [_med_rpps("xdp", r) for r in rates]
print("hal_return_pps=", [None if v is None else round(v) for v in hali_rpps])
print("xdp_return_pps=", [None if v is None else round(v) for v in xdp_rpps])

fig, _ = figure_tx_rtt_return_pps(
    rates_pps=rates,
    hali_return_pps=hal_rpps,
    xdp_return_pps=xdp_rpps,
    title="Return PPS vs send rate",
)
display(fig)
plt.close(fig)
